In [1]:
import os, time, requests, json, pandas as pd
from datetime import datetime, timedelta
from dotenv import load_dotenv
from google.analytics.data_v1beta import BetaAnalyticsDataClient
from googleapiclient.discovery import build
from google.analytics.data_v1beta.types import (
    DateRange,
    Dimension,
    Metric,
    RunReportRequest,
)
CLARITY_TOKEN = os.getenv("CLARITY_TOKEN")
load_dotenv()
PROPERTY_ID = os.getenv("PROPERTY_ID")
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = 'credentials.json'
client = BetaAnalyticsDataClient()

In [ ]:
def fetchGA4ClarityData(start_date: str, end_date: str):
    """
    Retrieves a comprehensive report based on the specified start and end dates.
    Args:
        start_date (str): The start date of the report in the format "YYYY-MM-DD".
        end_date (str): The end date of the report in the format "YYYY-MM-DD".
    """
    
    # Fetch GA4 Data
    request3 = RunReportRequest(
        property=f"properties/{PROPERTY_ID}",
        dimensions=[Dimension(name="eventName")],
        metrics=[Metric(name="eventCount")],
        date_ranges=[DateRange(start_date=start_date, end_date=end_date)],
    )

    response3 = client.run_report(request3)

    result = "**User Interaction Metrics:**\n"
    scroll = ""
    for row in response3.rows:
        event_name = row.dimension_values[0].value
        event_count = row.metric_values[0].value
        if event_name == "scroll":
            scroll = f"Scroll Count: {event_count} times"
        else:
            result += f"{event_name.capitalize()}: {event_count}\n"

    # Fetch Clarity Data
    params = {"numOfDays": "3", "dimension1": "PopularPages"}
    headers = {
        "Authorization": f"Bearer {CLARITY_TOKEN}",
        "Content-type": "application/json"
    }
    response = requests.get(
        "https://www.clarity.ms/export-data/api/v1/project-live-insights",
        params=params,
        headers=headers,
    )

    if response.status_code != 200:
        return result

    # Dead Clicks and Scroll Depth
    for metric in response.json():
        metric_name = metric.get('metricName', 'Unknown Metric')
        information = metric.get('information', [{}])

        if len(information) == 1:
            information = information[0]
            if metric_name == "DeadClickCount":
                result += f"**Dead Clicks:**\n"
                result += f"Total Dead Clicks: {information.get('subTotal', 'N/A')}\n"
                result += f"Sessions with Dead Clicks: {information.get('sessionsWithMetricPercentage', 'N/A')}%\n"
                result += f"Sessions without Dead Clicks: {information.get('sessionsWithoutMetricPercentage', 'N/A')}%\n"

            elif metric_name == "ScrollDepth":
                result += f"**Scroll Metrics:**\n"
                result += f"{scroll}\n"
                result += f"Average Scroll Depth: {information.get('averageScrollDepth', 'N/A')}%\n"

            elif metric_name == "Traffic":
                result += f"**Traffic and Session Metrics:**\n"
                result += f"Total Session Count: {information.get('totalSessionCount', 'N/A')}\n"
                result += f"Total Bot Session Count: {information.get('totalBotSessionCount', 'N/A')}\n"
                result += f"Distinct User Count: {information.get('distinctUserCount', 'N/A')}\n"
                result += f"Pages Per Session Percentage: {information.get('pagesPerSessionPercentage', 'N/A'):.2f}%\n"

            elif metric_name == "EngagementTime":
                result += f"**Engagement Time:**\n"
                result += f"Total Time: {information.get('totalTime', 'N/A')} seconds\n"
                result += f"Active Time: {information.get('activeTime', 'N/A')} seconds\n"

        elif len(information) > 1 and metric_name == "PageTitle":
            result += f"**Page Titles:**\n"
            for entry in information:
                if isinstance(entry, dict):
                    result += f"\"{entry.get('name', 'N/A')}\" - {entry.get('sessionsCount', 'N/A')} sessions\n"

    return result



In [41]:
date_range = DateRange(start_date="2024-01-01", end_date="2024-11-01")

request1 = RunReportRequest(
    property=f"properties/{PROPERTY_ID}",
    dimensions=[Dimension(name="eventName")],
    metrics=[Metric(name="eventCount")],
    date_ranges=[date_range],
)

report1 = client.run_report(request1)

output_string = ""
for row in report1.rows:
    event_name = row.dimension_values[0].value
    event_count = row.metric_values[0].value
    if event_name == "scroll":
        output_string += f"{event_name}: {event_count} times\n"
    else:
        output_string += f"{event_name.capitalize()}: {event_count}\n"

request2 = RunReportRequest(
    property=f"properties/{PROPERTY_ID}",
    dimensions=[
        Dimension(name="newVsReturning")
    ],
    metrics=[
        Metric(name="activeUsers")
    ],
    date_ranges=[
        DateRange(date_range)
    ]
)

report2 = client.run_report(request2)

new_visitors = 0
returning_visitors = 0

for row in report2.rows:
    visitor_type = row.dimension_values[0].value
    active_users = int(row.metric_values[0].value)
    
    if visitor_type == "new":
        new_visitors += active_users
    elif visitor_type == "returning":
        returning_visitors += active_users

total_visitors = new_visitors + returning_visitors

if total_visitors > 0:
    new_visitor_percentage = (new_visitors / total_visitors) * 100
    returning_visitor_percentage = (returning_visitors / total_visitors) * 100
else:
    new_visitor_percentage = 0
    returning_visitor_percentage = 0

output_string += f"New Visitors: {new_visitors}\nReturning Visitors: {returning_visitors}\nNew Visitors: {new_visitor_percentage:.2f}%\nReturning Visitors: {returning_visitor_percentage:.2f}%"

metrics = [
    Metric(name="activeUsers"),
    Metric(name="sessions"),
    Metric(name="averageSessionDuration"),
    Metric(name="sessionsPerUser"),
    Metric(name="scrolledUsers"),
    Metric(name="engagedSessions"),
    Metric(name="engagementRate"),
    Metric(name="addToCarts"),
    Metric(name="checkouts"),
    Metric(name="purchaserRate"),
]

request3 = RunReportRequest(
    property=f"properties/{PROPERTY_ID}",
    metrics=metrics,
    date_ranges=[date_range]
)
report3 = client.run_report(request3)

# Iterate through the rows and append the metric names and values to the string
for row in report3.rows:
    for i, metric_value in enumerate(row.metric_values):
        metric_name = report3.metric_headers[i].name
        output_string += f"{metric_name}: {metric_value.value}\n"

print(output_string)

Page_view: 23106
User_engagement: 16876
Session_start: 5026
First_visit: 4468
scroll: 2390 times
Form_start: 903
Form_submit: 807
View_search_results: 539
Click: 58
New Visitors: 3451
Returning Visitors: 177
New Visitors: 95.12%
Returning Visitors: 4.88%activeUsers: 3524
sessions: 8684
averageSessionDuration: 107.42320683440811
sessionsPerUser: 2.4642451759364357
scrolledUsers: 1554
engagedSessions: 6828
engagementRate: 0.786273606632888
addToCarts: 0
checkouts: 0
purchaserRate: 0



In [38]:
request = RunReportRequest(
    property=f"properties/{PROPERTY_ID}",
    dimensions=[
        Dimension(name="newVsReturning")
    ],
    metrics=[
        Metric(name="activeUsers")
    ],
    date_ranges=[
        DateRange(date_range)
    ]
)

response = client.run_report(request)

new_visitors = 0
returning_visitors = 0

for row in response.rows:
    visitor_type = row.dimension_values[0].value
    active_users = int(row.metric_values[0].value)
    
    if visitor_type == "new":
        new_visitors += active_users
    elif visitor_type == "returning":
        returning_visitors += active_users

total_visitors = new_visitors + returning_visitors

if total_visitors > 0:
    new_visitor_percentage = (new_visitors / total_visitors) * 100
    returning_visitor_percentage = (returning_visitors / total_visitors) * 100
else:
    new_visitor_percentage = 0
    returning_visitor_percentage = 0

result = f"New Visitors: {new_visitors}\nReturning Visitors: {returning_visitors}\nNew Visitors: {new_visitor_percentage:.2f}%\nReturning Visitors: {returning_visitor_percentage:.2f}%"
print(result)

New Visitors: 3451
Returning Visitors: 177
New Visitors: 95.12%
Returning Visitors: 4.88%


In [7]:
print(fetchGA4ClarityData("2024-01-01", "2024-11-01"))

**User Interaction Metrics:**
Page_view: 23106
User_engagement: 16876
Session_start: 5026
First_visit: 4468
Form_start: 903
Form_submit: 807
View_search_results: 539
Click: 58
**Dead Clicks:**
Total Dead Clicks: 13
Sessions with Dead Clicks: 16.07%
Sessions without Dead Clicks: 83.93%
**Scroll Metrics:**
Scroll Count: Scroll Count: 2390 times

Average Scroll Depth: 58.2%
**Traffic and Session Metrics:**
Total Session Count: 56
Total Bot Session Count: 22
Distinct User Count: 69
Pages Per Session Percentage: 1.66%
**Engagement Time:**
Total Time: 127 seconds
Active Time: 60 seconds
**Page Titles:**
"Apple Mac Mini MU9D3HN/A M4 Chip (16 GB RAM/10-core GPU Graphics/256 GB SSD Capacity/macOS Sequoia) Mini Tower - Egenie Stores" - 30 sessions
"Home - Egenie4u" - 17 sessions
"Checkout - Egenie Stores" - 4 sessions
"Laptops - Egenie Stores" - 4 sessions
"My account - Egenie Stores" - 4 sessions
"Lenovo LOQ 2024 Intel i5-13450HX - Egenie Stores" - 3 sessions
"ASUS TUF Gaming A16 Advantage Edit